# EasyRAG Naive Retrieval Basics

## Chapter position

This chapter starts once documents are already indexed. The focus is how a user question is prepared, recalled, filtered, fused, reranked, and finally turned into inspectable citations.

## Learning question

What does the naive retrieval path do before hybrid signals, graph neighbors, or rerank enter the picture?

## Success criteria

- run the simplest retrieval mode with all preprocessing disabled
- inspect naive chunk hits before hybrid fusion and graph signals enter the picture
- compare token fallback and dense retrieval behavior on the same corpus

## Source code anchors

- `easyrag.rag.retrieval.query_modes.naive_query`
- `easyrag.rag.types.QueryParam`
- `easyrag.rag.retrieval.pipeline.execute_query`


## Method principles

- `easyrag.rag.retrieval.query_modes.naive_query`: This is the simplest retrieval mode. It searches the chunk namespace directly without graph expansion, summary-based context, or multi-mode fusion.
- `easyrag.rag.types.QueryParam`: This dataclass is the retrieval control surface. It does not run retrieval itself; it carries mode, top-k, rewrite, MQE, rerank, and filter policy into the pipeline.
- `easyrag.rag.retrieval.pipeline.execute_query`: This is the end-to-end retrieval pipeline. It runs preparation, candidate generation, filtering, hydration, optional rerank, and result assembly into one `QueryResult`.

Design reason: these anchors keep query preparation, candidate generation, ranking, and hydration separate. That separation is intentional because retrieval debugging becomes much more precise when each boundary has its own visible inputs and outputs.


## How the code fits together

The flow in this notebook is naive query -> raw chunk hits -> fallback backend comparison. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

Design reason: the notebook isolates query preparation before broadening into ranking, fusion, rerank, or hydration. That is why the intermediate metadata stays visible enough to explain recall and ranking changes instead of collapsing everything into one opaque search result.


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

## What this cell is proving

We start with the most stripped-down retrieval setting: naive mode, no rewrite, no MQE, and a tiny chunk budget.

Design reason: this cell crosses the boundary from prepared inputs into ranked evidence. The notebook uses the structured retrieval APIs on purpose so citations, mode choice, and query metadata stay inspectable instead of disappearing behind a single search string.


In [ ]:
naive_tmp = tempfile.TemporaryDirectory()
naive_rag = EasyRAG(
    working_dir=naive_tmp.name,
    workspace="naive-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(naive_rag.initialize_storages())
run_sync(
    naive_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses query rewrite to keep retrieval grounded.\n",
            "# Storage\nPacked evidence stays available for answer synthesis.\n",
        ],
        ids=["doc::retrieval", "doc::storage"],
        file_paths=["docs/retrieval.md", "docs/storage.md"],
    )
)
naive_result = run_sync(
    naive_rag.aquery(
        "How does EasyRAG keep retrieval grounded?",
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)
_print_json({"metadata": naive_result.metadata, "citations": naive_result.citations})

## Why this output looks like this

The next cell forces the token-overlap fallback path by using a failing embedding function. That makes the backend swap visible without changing the rest of the retrieval contract.

Design reason: the output keeps evidence and metadata together because retrieval is not only about the top text string. The notebook preserves ranking context, mode choice, and query-preparation fields so the next step can explain why these citations appeared and why others did not.


In [ ]:
def failing_embedding(texts):
    raise RuntimeError("dense embedding unavailable")


fallback_tmp = tempfile.TemporaryDirectory()
fallback_rag = EasyRAG(
    working_dir=fallback_tmp.name,
    workspace="naive-fallback-demo",
    embedding_func=failing_embedding,
    query_model_func=_stub_query_model,
)
run_sync(fallback_rag.initialize_storages())
run_sync(
    fallback_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses query rewrite to keep retrieval grounded.\n",
            "# Storage\nPacked evidence stays available for answer synthesis.\n",
        ],
        ids=["doc::retrieval", "doc::storage"],
        file_paths=["docs/retrieval.md", "docs/storage.md"],
    )
)
fallback_result = run_sync(
    fallback_rag.aquery(
        "query rewrite",
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)
_print_json(
    {"metadata": fallback_result.metadata, "citations": fallback_result.citations}
)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

If the environment is configured, the optional cell shows the naive path with the repo's default provider wiring.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-naive-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                "# Retrieval\nQuery rewrite keeps retrieval grounded.",
                ids=["doc::retrieval"],
                file_paths=["docs/retrieval.md"],
            )
        )
        provider_result = run_sync(
            provider_rag.aquery(
                "query rewrite",
                QueryParam(
                    mode="naive",
                    rewrite_enabled=False,
                    mqe_enabled=False,
                    chunk_top_k=2,
                ),
            )
        )
        _print_json(provider_result.metadata)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- Keep query preparation, candidate generation, fusion, rerank, and hydration separate in your head. They fail in different ways.
- A weak citation list can come from filtering, ranking, or missing hydration, not only from poor recall.
- `QueryResult.metadata` is usually the fastest way to see what the pipeline actually did.

## Takeaway

Naive retrieval keeps the search path narrow: one query, one chunk namespace, one final ranked list. That simplicity is valuable because it tells you whether later hybrid behavior is helping or only hiding a weak baseline.

In [ ]:
run_sync(naive_rag.finalize_storages())
naive_tmp.cleanup()
run_sync(fallback_rag.finalize_storages())
fallback_tmp.cleanup()
print("Cleaned up the naive-retrieval workspaces.")

## Where to go next

- Continue with [04_04_hybrid_metadata_filter_and_modes.ipynb](04_04_hybrid_metadata_filter_and_modes.ipynb) to see how the baseline changes once more retrieval signals join.
- Read [engineering/22-retrieval-pipeline.md](../../docs/engineering/22-retrieval-pipeline.md) for the code-oriented retrieval walkthrough.
- Revisit [04_02_query_rewrite_and_multi_query.ipynb](04_02_query_rewrite_and_multi_query.ipynb) if you want the prepared-query view before ranking.